# 01 - Data Acquisition

## CyberForge AI - Cybersecurity Data Collection & Preparation

This notebook handles all data acquisition for the CyberForge AI ML pipeline.

### Data Sources:
1. **Public Datasets** - Legal, publicly available cybersecurity datasets
2. **Web Scraper API** - Real-time website security data collection
3. **Hugging Face Datasets** - Pre-uploaded CyberForge datasets
4. **Synthetic Data** - Generated data for edge cases

### Output:
- Cleaned, normalized datasets ready for feature engineering
- Data validation reports

In [ ]:
# Load configuration from environment setup
import json
import pandas as pd
import numpy as np
from pathlib import Path
import httpx
import asyncio
from datetime import datetime
from typing import Dict, List, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Load notebook config (same directory as this notebook)
config_path = Path("notebook_config.json")
if not config_path.exists():
    # Try absolute path as fallback
    config_path = Path("/home/user/app/notebooks/notebook_config.json")
    
if config_path.exists():
    with open(config_path) as f:
        CONFIG = json.load(f)
    print("✓ Configuration loaded")
else:
    raise FileNotFoundError("Run 00_environment_setup.ipynb first!")

DATASETS_DIR = Path(CONFIG.get("datasets_dir", "/home/user/app/datasets"))
print(f"✓ Datasets directory: {DATASETS_DIR}")

## 1. Web Scraper API Data Collection

Collect real-time website security data using the WebScrapper.live API.

In [ ]:
class WebScraperDataCollector:
    """
    Collects website security data via WebScrapper.live API.
    Uses synchronous httpx for Jupyter compatibility.
    """
    
    def __init__(self):
        self.api_url = "http://webscrapper.live/api/scrape"
        self.api_key = "sk-fd14eaa7bceb478db7afc7256e514d2b"
        self.timeout = 60.0
    
    def scrape_website(self, url: str) -> Dict[str, Any]:
        """Scrape a single website and return security data (sync version)"""
        try:
            with httpx.Client(timeout=self.timeout) as client:
                response = client.post(
                    self.api_url,
                    json={"url": url},
                    headers={
                        "Content-Type": "application/json",
                        "X-API-Key": self.api_key
                    }
                )
                
                if response.status_code == 200:
                    data = response.json()
                    return {
                        "success": True,
                        "url": url,
                        "security_report": data.get("security_report", {}),
                        "network_requests": data.get("network_requests", []),
                        "console_logs": data.get("console_logs", []),
                        "performance_metrics": data.get("performance_metrics", {}),
                        "response_headers": data.get("response_headers", {}),
                        "html_length": len(data.get("html", "") or data.get("content", "")),
                        "scraped_at": datetime.now().isoformat()
                    }
                else:
                    return {"success": False, "url": url, "error": f"Status {response.status_code}"}
        except Exception as e:
            return {"success": False, "url": url, "error": str(e)}
    
    def collect_batch(self, urls: List[str]) -> List[Dict]:
        """Collect data from multiple URLs (sync version)"""
        import time
        results = []
        for url in urls:
            print(f"  Scraping: {url}")
            result = self.scrape_website(url)
            results.append(result)
            time.sleep(1)  # Rate limiting
        return results
    
    def extract_security_features(self, data: Dict) -> Dict:
        """Extract security-relevant features from scraped data"""
        if not data.get("success"):
            return None
        
        security_report = data.get("security_report", {})
        network_requests = data.get("network_requests", [])
        console_logs = data.get("console_logs", [])
        
        return {
            "url": data["url"],
            "is_https": security_report.get("is_https", False),
            "has_mixed_content": security_report.get("mixed_content", False),
            "missing_headers_count": len(security_report.get("missing_security_headers", [])),
            "has_insecure_cookies": security_report.get("insecure_cookies", False),
            "total_requests": len(network_requests),
            "external_requests": sum(1 for r in network_requests if self._is_external(r, data["url"])),
            "failed_requests": sum(1 for r in network_requests if r.get("status", 200) >= 400),
            "console_errors": sum(1 for log in console_logs if log.get("level") == "error"),
            "console_warnings": sum(1 for log in console_logs if log.get("level") == "warning"),
            "html_size": data.get("html_length", 0),
            "scraped_at": data["scraped_at"]
        }
    
    def _is_external(self, request: Dict, base_url: str) -> bool:
        """Check if a request is to an external domain"""
        try:
            from urllib.parse import urlparse
            base_domain = urlparse(base_url).netloc
            req_domain = urlparse(request.get("url", "")).netloc
            return base_domain != req_domain
        except:
            return False

scraper = WebScraperDataCollector()
print("✓ Web Scraper Data Collector initialized")


In [ ]:
# Collect sample data from known safe/unsafe websites for training
SAMPLE_URLS = [
    # Known safe websites
    "https://www.google.com",
    "https://github.com",
    "https://www.microsoft.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    # Test sites
    "https://example.com",
    "https://httpbin.org",
]

print(f"Collecting data from {len(SAMPLE_URLS)} URLs...")

# Run synchronous collection (Jupyter-compatible)
scraped_data = scraper.collect_batch(SAMPLE_URLS[:3])  # Limited for demo

# Extract features
features_list = [scraper.extract_security_features(d) for d in scraped_data if d.get("success")]
if features_list:
    web_scraper_df = pd.DataFrame(features_list)
    print(f"\n✓ Collected {len(web_scraper_df)} website security profiles")
    display(web_scraper_df.head())
else:
    print("⚠ No data collected - API may be unavailable")
    web_scraper_df = pd.DataFrame()

## 2. Load Hugging Face Datasets

Load pre-uploaded CyberForge datasets from Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import os

HF_DATASET_REPO = "Che237/cyberforge-datasets"

def list_available_datasets():
    """List all available datasets in the HF repository"""
    try:
        files = list_repo_files(HF_DATASET_REPO, repo_type="dataset")
        csv_files = [f for f in files if f.endswith('.csv')]
        return csv_files
    except Exception as e:
        print(f"⚠ Could not list HF datasets: {e}")
        return []

def download_dataset(file_path: str, local_dir: Path = DATASETS_DIR) -> Optional[Path]:
    """Download a specific dataset from Hugging Face"""
    try:
        local_path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=file_path,
            repo_type="dataset",
            cache_dir=str(local_dir / "cache")
        )
        return Path(local_path)
    except Exception as e:
        print(f"⚠ Could not download {file_path}: {e}")
        return None

# List available datasets
print("Available datasets on Hugging Face:")
available_files = list_available_datasets()
for f in available_files[:20]:  # Show first 20
    print(f"  - {f}")
print(f"  ... and {len(available_files) - 20} more" if len(available_files) > 20 else "")

In [ ]:
# Priority datasets for training (aligned with backend requirements)
PRIORITY_DATASETS = {
    "network_intrusion": "network_intrusion/network_intrusion_processed.csv",
    "phishing_detection": "phishing_detection/phishing_detection_processed.csv",
    "malware_detection": "malware_detection/malware_detection_processed.csv",
    "anomaly_detection": "anomaly_detection/anomaly_detection_processed.csv",
    "web_attack_detection": "web_attack_detection/web_attack_detection_processed.csv",
}

loaded_datasets = {}

print("Downloading priority datasets...")
for name, path in PRIORITY_DATASETS.items():
    if path in available_files:
        print(f"  Downloading: {name}")
        local_path = download_dataset(path)
        if local_path:
            try:
                df = pd.read_csv(local_path)
                loaded_datasets[name] = df
                print(f"    ✓ {name}: {len(df)} samples, {len(df.columns)} features")
            except Exception as e:
                print(f"    ⚠ Could not load {name}: {e}")
    else:
        print(f"  ⚠ {name}: Not found in repository")

print(f"\n✓ Loaded {len(loaded_datasets)} datasets")

## 3. Load Local Datasets

Load any datasets already present in the local datasets directory.

In [ ]:
def load_local_datasets(datasets_dir: Path) -> Dict[str, pd.DataFrame]:
    """Load all CSV datasets from local directory"""
    datasets = {}
    
    for csv_file in datasets_dir.rglob("*_processed.csv"):
        try:
            name = csv_file.stem.replace("_processed", "")
            df = pd.read_csv(csv_file)
            datasets[name] = df
            print(f"  ✓ {name}: {len(df)} samples")
        except Exception as e:
            print(f"  ⚠ {csv_file.name}: {e}")
    
    return datasets

print("Loading local datasets...")
local_datasets = load_local_datasets(DATASETS_DIR)

# Merge with HF datasets (local takes precedence)
for name, df in local_datasets.items():
    if name not in loaded_datasets:
        loaded_datasets[name] = df

print(f"\n✓ Total datasets available: {len(loaded_datasets)}")

## 4. Data Validation & Quality Checks

In [ ]:
def validate_dataset(name: str, df: pd.DataFrame) -> Dict[str, Any]:
    """Validate dataset quality and return report"""
    # Expanded label column detection
    label_columns = ['label', 'target', 'class', 'is_malicious', 'attack_type', 
                     'attack', 'category', 'malware', 'phishing', 'threat', 
                     'classification', 'type', 'result', 'output', 'y']
    has_label = any(col.lower() in [c.lower() for c in df.columns] for col in label_columns)
    
    report = {
        "name": name,
        "samples": len(df),
        "features": len(df.columns),
        "missing_values": df.isnull().sum().sum(),
        "missing_pct": (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100 if len(df) > 0 else 0,
        "duplicate_rows": df.duplicated().sum(),
        "numeric_columns": len(df.select_dtypes(include=[np.number]).columns),
        "categorical_columns": len(df.select_dtypes(include=['object', 'category']).columns),
        "memory_mb": df.memory_usage(deep=True).sum() / (1024 * 1024),
        "has_label": has_label,
        "valid": True
    }
    
    # More lenient validation - only fail on critical issues
    issues = []
    if report["samples"] < 10:
        issues.append("Too few samples (<10)")
    if report["missing_pct"] > 80:
        issues.append("Too many missing values (>80%)")
    
    report["issues"] = issues
    report["valid"] = len(issues) == 0
    
    return report

# Validate all datasets
validation_reports = []
print("Validating datasets...\n")
print(f"{'Dataset':<30} {'Samples':>10} {'Features':>10} {'Missing %':>10} {'Valid':>8}")
print("-" * 75)

for name, df in loaded_datasets.items():
    report = validate_dataset(name, df)
    validation_reports.append(report)
    status = "✓" if report["valid"] else "⚠"
    print(f"{name:<30} {report['samples']:>10} {report['features']:>10} {report['missing_pct']:>9.2f}% {status:>8}")

valid_datasets = [r["name"] for r in validation_reports if r["valid"]]
print(f"\n✓ {len(valid_datasets)} datasets passed validation")


## 5. Data Normalization

In [ ]:
def normalize_dataset(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Normalize dataset for consistent processing"""
    df = df.copy()
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
    
    # Find and standardize label column (expanded list)
    label_columns = ['label', 'target', 'class', 'is_malicious', 'attack_type', 
                     'attack', 'category', 'malware', 'phishing', 'threat',
                     'classification', 'type', 'result', 'output', 'y']
    for col in label_columns:
        matching_cols = [c for c in df.columns if c.lower() == col.lower()]
        if matching_cols:
            df = df.rename(columns={matching_cols[0]: 'label'})
            break
    
    # Handle missing values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        if col != 'label':
            df[col] = df[col].fillna('unknown')
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Add metadata
    df.attrs['dataset_name'] = name
    df.attrs['processed_at'] = datetime.now().isoformat()
    
    return df

# Normalize ALL loaded datasets (not just valid ones)
normalized_datasets = {}
print("Normalizing datasets...")

for name, df in loaded_datasets.items():
    try:
        normalized_df = normalize_dataset(df, name)
        normalized_datasets[name] = normalized_df
        print(f"  ✓ {name}: {len(normalized_df)} samples after normalization")
    except Exception as e:
        print(f"  ⚠ {name}: Error during normalization - {e}")

print(f"\n✓ Normalized {len(normalized_datasets)} datasets")


## 6. Save Processed Data

In [ ]:
# Create processed data directory
PROCESSED_DIR = DATASETS_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

# Save each normalized dataset
print("Saving processed datasets...")
dataset_manifest = []

for name, df in normalized_datasets.items():
    output_path = PROCESSED_DIR / f"{name}_ready.csv"
    df.to_csv(output_path, index=False)
    
    manifest_entry = {
        "name": name,
        "path": str(output_path.relative_to(DATASETS_DIR.parent)),
        "samples": len(df),
        "features": len(df.columns),
        "has_label": "label" in df.columns,
        "processed_at": datetime.now().isoformat()
    }
    dataset_manifest.append(manifest_entry)
    print(f"  ✓ Saved: {output_path.name}")

# Save manifest
manifest_path = PROCESSED_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(dataset_manifest, f, indent=2)

print(f"\n✓ Dataset manifest saved to: {manifest_path}")

## 7. Summary

In [ ]:
print("\n" + "=" * 60)
print("DATA ACQUISITION COMPLETE")
print("=" * 60)

total_samples = sum(len(df) for df in normalized_datasets.values())

print(f"""
📊 Data Collection Summary:
   - Datasets processed: {len(normalized_datasets)}
   - Total samples: {total_samples:,}
   - Output directory: {PROCESSED_DIR}

📁 Datasets Ready for Feature Engineering:""")

for entry in dataset_manifest:
    print(f"   ✓ {entry['name']}: {entry['samples']:,} samples")

print(f"""
Next step:
  → 02_feature_engineering.ipynb
""")
print("=" * 60)